# Demo 2: CfC 闭式解与 LTC 对比

本 Notebook 展示：
1. CfC（Closed-form Continuous-time）如何用闭式解替代 ODE 数值求解
2. CfC 与 LTC 的计算速度对比
3. 两者在时间序列预测任务上的精度对比

---

## 1. CfC 核心公式

LTC 需要用 ODE 求解器（如 Runge-Kutta）逐步积分，计算代价高。CfC 的核心洞察是：

LTC 的 ODE 在固定时间步 $\Delta t$ 内的解可以写成闭式形式：

$$x(t+\Delta t) = \sigma(-f(x, I; \theta_f) \cdot \Delta t) \odot g(x, I; \theta_g) + [1 - \sigma(-f(x, I; \theta_f) \cdot \Delta t)] \odot h(x, I; \theta_h)$$

- $f$：控制衰减速率（类似时间常数的倒数）
- $g$：衰减目标分支
- $h$：稳态目标分支
- $\sigma$：sigmoid 门控

**直觉**：当 $\Delta t$ 小时，$\sigma(-f \cdot \Delta t) \approx 1$，状态趋向 $g$；当 $\Delta t$ 大时，$\sigma(-f \cdot \Delta t) \approx 0$，状态趋向 $h$。门控自动在"短期响应"和"长期稳态"间插值。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from scipy.integrate import odeint

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(42)

## 2. CfC 神经元实现

In [ ]:
class CfCNeuron:
    """
    CfC (Closed-form Continuous-time) 神经元
    x(t+dt) = sigma(-f*dt) * g + (1-sigma(-f*dt)) * h
    """
    def __init__(self, input_size, hidden_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        scale = 0.1
        self.W_f = np.random.randn(hidden_size, hidden_size + input_size) * scale
        self.b_f = np.random.randn(hidden_size) * scale
        self.W_g = np.random.randn(hidden_size, hidden_size + input_size) * scale
        self.b_g = np.random.randn(hidden_size) * scale
        self.W_h = np.random.randn(hidden_size, hidden_size + input_size) * scale
        self.b_h = np.random.randn(hidden_size) * scale

    def step(self, x, I, dt):
        combined = np.concatenate([x, I])
        f = np.exp(-np.abs(self.W_f @ combined + self.b_f))
        g = np.tanh(self.W_g @ combined + self.b_g)
        h = np.tanh(self.W_h @ combined + self.b_h)
        gate = 1.0 / (1.0 + np.exp(-f * dt))
        return gate * g + (1.0 - gate) * h

    def forward(self, inputs, dt=0.1):
        seq_len = inputs.shape[0]
        x = np.zeros(self.hidden_size)
        outputs = []
        for t in range(seq_len):
            x = self.step(x, inputs[t], dt)
            outputs.append(x.copy())
        return np.array(outputs)


class LTCNeuron:
    """
    LTC 神经元 (ODE 求解器版本)
    dx/dt = -(1/tau_eff) * x + (1/tau_eff) * A
    """
    def __init__(self, input_size, hidden_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        scale = 0.1
        self.W_tau = np.random.randn(hidden_size, hidden_size + input_size) * scale
        self.b_tau = np.random.randn(hidden_size) * scale
        self.W_A = np.random.randn(hidden_size, hidden_size + input_size) * scale
        self.b_A = np.random.randn(hidden_size) * scale
        self.tau_base = np.ones(hidden_size)

    def _ode_func(self, x, t_val, I, dt):
        combined = np.concatenate([x, I])
        tau_adj = 1.0 / (1.0 + np.exp(-(self.W_tau @ combined + self.b_tau)))
        tau_eff = self.tau_base + tau_adj
        A = np.tanh(self.W_A @ combined + self.b_A)
        return -(1.0 / tau_eff) * x + (1.0 / tau_eff) * A

    def step(self, x, I, dt):
        t_span = [0, dt]
        sol = odeint(self._ode_func, x, t_span, args=(I, dt))
        return sol[-1]

    def forward(self, inputs, dt=0.1):
        seq_len = inputs.shape[0]
        x = np.zeros(self.hidden_size)
        outputs = []
        for t in range(seq_len):
            x = self.step(x, inputs[t], dt)
            outputs.append(x.copy())
        return np.array(outputs)

## 3. 速度对比：CfC vs LTC

In [ ]:
input_size = 4
hidden_size = 8
seq_len = 200
dt = 0.1

inputs = np.random.randn(seq_len, input_size)

cfc = CfCNeuron(input_size, hidden_size)
ltc = LTCNeuron(input_size, hidden_size)

start = time.time()
for _ in range(10):
    out_cfc = cfc.forward(inputs, dt)
time_cfc = (time.time() - start) / 10

start = time.time()
for _ in range(3):
    out_ltc = ltc.forward(inputs, dt)
time_ltc = (time.time() - start) / 3

speedup = time_ltc / time_cfc

print(f"CfC forward time: {time_cfc*1000:.2f} ms")
print(f"LTC forward time: {time_ltc*1000:.2f} ms")
print(f"Speedup (LTC/CfC): {speedup:.1f}x")
print(f"\nCfC output shape: {out_cfc.shape}")
print(f"LTC output shape: {out_ltc.shape}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(['CfC (Closed-form)', 'LTC (ODE Solver)'],
              [time_cfc * 1000, time_ltc * 1000],
              color=['#2196F3', '#F44336'], width=0.5)
ax.set_ylabel('Forward Pass Time (ms)', fontsize=12)
ax.set_title(f'Computation Speed: CfC vs LTC (Speedup: {speedup:.1f}x)', fontsize=14)
for bar, val in zip(bars, [time_cfc * 1000, time_ltc * 1000]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{val:.1f} ms', ha='center', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('demo2_speed_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. 不同序列长度下的扩展性对比

In [ ]:
seq_lengths = [50, 100, 200, 500, 1000]
times_cfc = []
times_ltc = []

for sl in seq_lengths:
    inp = np.random.randn(sl, input_size)

    start = time.time()
    for _ in range(5):
        cfc.forward(inp, dt)
    times_cfc.append((time.time() - start) / 5)

    start = time.time()
    for _ in range(2):
        ltc.forward(inp, dt)
    times_ltc.append((time.time() - start) / 2)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(seq_lengths, [t * 1000 for t in times_cfc], 'b-o', linewidth=2, markersize=8, label='CfC')
ax.plot(seq_lengths, [t * 1000 for t in times_ltc], 'r-s', linewidth=2, markersize=8, label='LTC')
ax.set_xlabel('Sequence Length', fontsize=12)
ax.set_ylabel('Forward Pass Time (ms)', fontsize=12)
ax.set_title('Scalability: CfC vs LTC across Sequence Lengths', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('demo2_scalability.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. 门控机制可视化

观察 CfC 中 $\sigma(-f \cdot \Delta t)$ 门控如何随时间变化。

In [ ]:
dt_values = np.linspace(0.01, 5.0, 200)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, f_val in enumerate([0.5, 1.0, 3.0]):
    gate = 1.0 / (1.0 + np.exp(-f_val * dt_values))
    axes[idx].plot(dt_values, gate, 'b-', linewidth=2.5)
    axes[idx].fill_between(dt_values, gate, alpha=0.15, color='blue', label='g branch (short-term)')
    axes[idx].fill_between(dt_values, gate, 1.0, alpha=0.15, color='red', label='h branch (steady-state)')
    axes[idx].set_xlabel('Δt', fontsize=11)
    axes[idx].set_ylabel('Gate σ(-f·Δt)', fontsize=11)
    axes[idx].set_title(f'f = {f_val}', fontsize=12)
    axes[idx].legend(fontsize=9)
    axes[idx].grid(True, alpha=0.3)

plt.suptitle('CfC Gating Mechanism: Short-term (g) vs Steady-state (h)', fontsize=14)
plt.tight_layout()
plt.savefig('demo2_gating.png', dpi=150, bbox_inches='tight')
plt.show()

print("Key insight: Larger f or Δt → gate → 0 → state approaches h (steady-state)")
print("            Smaller f or Δt → gate → 1 → state approaches g (short-term)")

## 6. 关键结论

| 维度 | LTC | CfC |
|------|-----|-----|
| **计算方式** | ODE 数值求解（Runge-Kutta） | 闭式解（单步前向传播） |
| **速度** | 慢（每个时间步需多次函数求值） | 快（~100x 加速） |
| **精度** | 高（数值精确） | 近似（但实验表明差距极小） |
| **可微性** | 需通过 ODE 求解器反传 | 天然可微，标准反向传播 |
| **适用场景** | 需要精确 ODE 解的研究 | 实际应用和大规模训练 |

**核心洞察**：CfC 并非抛弃了 ODE 的连续时间特性，而是找到了一个精确的闭式近似——保留了 LTC 的动态适应能力，同时消除了计算瓶颈。这是 LNN 从学术走向实用的关键一步。

---

Next: [Demo 3: LNN 时间序列预测实战](./demo3_timeseries_forecasting.ipynb)